In [1]:
import pandas as pd
import sys
from pathlib import Path
current_path = Path().resolve()
sys.path.append(str(current_path.parent))

In [2]:
from src.data.load_data import load_orders, load_items, load_customers,load_sellers, load_category, load_geo, load_payments, load_products, load_reviews

orders = load_orders()
order_items = load_items()
customers = load_customers()
sellers = load_sellers()
category = load_category()
geo = load_geo()
payments = load_payments()
products = load_products()
reviews = load_reviews()

In [3]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [5]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

In [6]:
orders['order_id'].duplicated().sum()

np.int64(0)

In [7]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [8]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [9]:
order_items['order_id'].duplicated().sum()

np.int64(13984)

In [10]:
order_items['product_id'].duplicated().sum()

np.int64(79699)

In [11]:
order_items.shape

(112650, 7)

In [12]:
geo_unique = (
    geo.groupby('geolocation_zip_code_prefix', as_index=False)
    .agg({
        'geolocation_lat':'mean',
        'geolocation_lng':'mean',
        'geolocation_city':'first',
        'geolocation_state':'first'
    })
)

payments_agg = (
    payments.groupby('order_id', as_index=False)
    .agg({
        'payment_value':'sum',
        'payment_installments':'max',
        'payment_type':'first'
    })
)
reviews_agg = (
    reviews.sort_values('review_creation_date')
    .drop_duplicates(subset='order_id', keep='last')
)

In [20]:
df = pd.merge(orders, customers, how='inner', on='customer_id')
df = pd.merge(df, order_items, how='inner', on='order_id')
df = pd.merge(df, products, how='left', on='product_id')
df = pd.merge(df, sellers, how='left', on='seller_id')
df = pd.merge(df, payments_agg, how='left', on='order_id')
df = pd.merge(df, reviews_agg, how='left', on='order_id')
df = pd.merge(
    df, geo_unique, how='left',
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix'
).rename(columns={
    'geolocation_lat':'customer_lat',
    'geolocation_lng':'customer_lng'
})
df = pd.merge(
    df, geo_unique, how='left',
    left_on='seller_zip_code_prefix',
    right_on='geolocation_zip_code_prefix'
).rename(columns={
    'geolocation_lat':'seller_lat',
    'geolocation_lng':'seller_lng'
})
df = pd.merge(df, category, how='left', on='product_category_name')

In [14]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,customer_lat,customer_lng,geolocation_city_x,geolocation_state_x,geolocation_zip_code_prefix_y,seller_lat,seller_lng,geolocation_city_y,geolocation_state_y,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,-23.576983,-46.587161,sao paulo,SP,9350.0,-23.680729,-46.444238,maua,SP,housewares
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,-23.576983,-46.587161,sao paulo,SP,9350.0,-23.680729,-46.444238,maua,SP,housewares
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,-23.576983,-46.587161,sao paulo,SP,9350.0,-23.680729,-46.444238,maua,SP,housewares
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,-12.177924,-44.660711,barreiras,BA,31570.0,-19.807681,-43.980427,belo horizonte,MG,perfumery
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,-16.745150,-48.514783,vianopolis,GO,14840.0,-21.363502,-48.229601,guariba,SP,auto


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118310 entries, 0 to 118309
Data columns (total 50 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       118310 non-null  object 
 1   customer_id                    118310 non-null  object 
 2   order_status                   118310 non-null  object 
 3   order_purchase_timestamp       118310 non-null  object 
 4   order_approved_at              118295 non-null  object 
 5   order_delivered_carrier_date   117056 non-null  object 
 6   order_delivered_customer_date  115722 non-null  object 
 7   order_estimated_delivery_date  118310 non-null  object 
 8   customer_unique_id             118310 non-null  object 
 9   customer_zip_code_prefix       118310 non-null  int64  
 10  customer_city                  118310 non-null  object 
 11  customer_state                 118310 non-null  object 
 12  order_item_id                 

In [16]:
df['review_score'].describe()

count    117332.000000
mean          4.031390
std           1.387994
min           1.000000
25%           4.000000
50%           5.000000
75%           5.000000
max           5.000000
Name: review_score, dtype: float64

In [17]:
df['review_score'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 118310 entries, 0 to 118309
Series name: review_score
Non-Null Count   Dtype  
--------------   -----  
117332 non-null  float64
dtypes: float64(1)
memory usage: 924.4 KB


In [18]:
df['product_category_name_english'].unique()

array(['housewares', 'perfumery', 'auto', 'pet_shop', 'stationery', nan,
       'furniture_decor', 'office_furniture', 'garden_tools',
       'computers_accessories', 'bed_bath_table', 'toys',
       'construction_tools_construction', 'telephony', 'health_beauty',
       'electronics', 'baby', 'cool_stuff', 'watches_gifts',
       'air_conditioning', 'sports_leisure', 'books_general_interest',
       'small_appliances', 'food', 'luggage_accessories',
       'fashion_underwear_beach', 'christmas_supplies',
       'fashion_bags_accessories', 'musical_instruments',
       'construction_tools_lights', 'books_technical',
       'costruction_tools_garden', 'home_appliances', 'market_place',
       'agro_industry_and_commerce', 'party_supplies', 'home_confort',
       'cds_dvds_musicals', 'industry_commerce_and_business',
       'consoles_games', 'furniture_bedroom', 'construction_tools_safety',
       'fixed_telephony', 'drinks',
       'kitchen_dining_laundry_garden_furniture', 'fashion_sho

In [19]:
df.groupby('product_category_name_english')['review_score'].mean().sort_values(ascending=False)

product_category_name_english
cds_dvds_musicals            4.642857
fashion_childrens_clothes    4.500000
books_general_interest       4.438503
books_imported               4.419355
flowers                      4.419355
                               ...   
home_comfort_2               3.642857
fashion_male_clothing        3.548611
office_furniture             3.526791
diapers_and_hygiene          3.256410
security_and_services        2.500000
Name: review_score, Length: 71, dtype: float64